## Problem #2, SynCode

In [2]:
import re
from syncode import Syncode

In [3]:
syncode_grammar = r"""
start: "Operands: " NUMBER " and " NUMBER NEWLINE "Operator: " OPERATOR
OPERATOR: "+" | "-" | "*" | "/"
NEWLINE: "\n"
%import common.NUMBER
"""

syn_model = Syncode(
    model="microsoft/phi-2", 
    grammar=syncode_grammar,
    max_new_tokens=50
)

base_prompt = """Extract the floating point operands and the operator from the query.

Query: What is 327. multiplied by 11.0?
Operands: 327.0 and 11.0
Operator: *

Query: What is 45.1 plus 23.54?
Operands: 45.1 and 23.54
Operator: +

Query: What is 120 divided by 4.0?
Operands: 120.0 and 4.0
Operator: /

Query: What is 18.9 minus 3.2?
Operands: 18.9 and 3.2
Operator: -

Query: What is 5 multiplied by 0.6?
Operands: 5.0 and 0.6
Operator: *

Query: {query}
"""

def run_syncode(textual_query):
    prompt = base_prompt.format(query=textual_query)
    
    raw_output = syn_model.infer(prompt)
    
    # string-ify the output (can be a list or dict or smth like that)
    output = raw_output[0] if isinstance(raw_output, list) else str(raw_output)

    # use regex to get all the numbers/operaands
    # whatever's left is the operator
    operand_matches = re.findall(r"-?\d+\.\d+|-?\d+", output)
    operands = [float(operand_matches[0]), float(operand_matches[1])]
    
    operator = ""
    if "+" in output: 
        operator = "+"
    elif "-" in output: 
        operator = "-"
    elif "*" in output: 
        operator = "*"
    elif "/" in output: 
        operator = "/"
        
    # use the extracted results to calc the final answer
    eval_string = f"{operands[0]} {operator} {operands[1]}"
    final_answer = float(eval(eval_string))
    
    return (operands, operator, final_answer)



[2026-10-28 03:36:32,536-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16
[2026-10-28 03:36:36,063-accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
print(run_syncode("What is 87.2 divided by 13.6?"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


([87.2, 13.6], '/', 6.411764705882353)
